# Benchmark Completo — Transfer Learning no CIFAR-10
**Versão Professor · Subset 5.000 imagens · 10 épocas · GPU**

Roda os 5 experimentos em sequência, do mais simples ao mais robusto, e gera uma tabela comparativa final.

| # | Estratégia | Params treináveis | Acurácia esperada (CPU 5ep) |
|---|---|---|---|
| 1 | Scratch Padrão | 11.2M | ~37.8% |
| 2 | Scratch + Augmentation | 11.2M | ~44.0% |
| 3 | Feature Extraction | 5.1K | ~74.5% |
| 4 | Fine-Tuning Parcial (layer4) | 2.6M | ~75.4% |
| 5 | Fine-Tuning Completo | 11.2M | ~87.9% |

> Com GPU T4 e 10 épocas, tempo estimado total: **~30–40 min**.

## ▶️ Passo 0 — Ativar a GPU

**Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)**

Execute a célula abaixo para confirmar.

In [1]:
import torch

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'✅ GPU ativa: {torch.cuda.get_device_name(0)}')
else:
    device = torch.device('cpu')
    print('⚠️  GPU NÃO detectada — ative em: Ambiente de execução → Alterar o tipo → GPU (T4)')
    print('   Sem GPU os experimentos levam várias horas.')

✅ GPU ativa: Tesla T4


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import Subset, DataLoader
import numpy as np
import time

torch.manual_seed(42)

NUM_EPOCHS = 10
N_TRAIN    = 5000
N_VAL      = 1000
BATCH      = 64

results = {}   # acumula resultados de todos os experimentos
print(f'Config: {N_TRAIN} treino | {N_VAL} val | {NUM_EPOCHS} épocas | batch={BATCH} | device={device}')

Config: 5000 treino | 1000 val | 10 épocas | batch=64 | device=cuda


In [3]:
# ── Transforms ─────────────────────────────────────────────────────────────
transform_std = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def extract_balanced_subset(dataset, n_total):
    n_per_class = n_total // 10
    indices, counts = [], {c: 0 for c in range(10)}
    for idx, label in enumerate(dataset.targets):
        if counts[label] < n_per_class:
            indices.append(idx)
            counts[label] += 1
        if len(indices) == n_total:
            break
    return Subset(dataset, indices)

# ── Datasets padrão (usados na maioria dos experimentos) ───────────────────
train_full_std = datasets.CIFAR10(root='data', train=True,  download=True, transform=transform_std)
val_full       = datasets.CIFAR10(root='data', train=False, download=True, transform=transform_std)

train_ds_std = extract_balanced_subset(train_full_std, N_TRAIN)
val_ds       = extract_balanced_subset(val_full, N_VAL)

train_loader_std = DataLoader(train_ds_std, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_loader       = DataLoader(val_ds,       batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

# ── Dataset com augmentation (Experimento 2) ───────────────────────────────
train_full_aug  = datasets.CIFAR10(root='data', train=True, download=True, transform=transform_aug)
train_ds_aug    = extract_balanced_subset(train_full_aug, N_TRAIN)
train_loader_aug = DataLoader(train_ds_aug, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)

print(f'✅ Dados carregados: {N_TRAIN} treino | {N_VAL} val')

100%|██████████| 170M/170M [00:04<00:00, 34.2MB/s]


✅ Dados carregados: 5000 treino | 1000 val


In [4]:
criterion = nn.CrossEntropyLoss()

def train_epoch(model, loader, optimizer):
    model.train()
    running_loss = 0.0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    return running_loss / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            _, preds = torch.max(model(inputs), 1)
            correct += (preds == labels).sum().item()
    return correct / len(loader.dataset)

def run_experiment(name, model, optimizer, train_loader, n_epochs=NUM_EPOCHS):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'\n{"="*55}')
    print(f'  {name}')
    print(f'  Parâmetros treináveis: {trainable:,}')
    print(f'{"="*55}')
    t0 = time.time()
    acc = 0.0
    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(model, train_loader, optimizer)
        acc  = evaluate(model, val_loader)
        print(f'  Época {epoch:>2}/{n_epochs} | Loss: {loss:.4f} | Val Acc: {acc*100:.2f}%')
    elapsed = time.time() - t0
    print(f'  ✅ {elapsed/60:.1f} min | Acurácia final: {acc*100:.2f}%')
    results[name] = {'acc': acc * 100, 'time_min': elapsed / 60, 'params': trainable}
    return model

print('✅ Funções auxiliares definidas.')

✅ Funções auxiliares definidas.


## Experimento 1 — Scratch Padrão
ResNet-18 com pesos **aleatórios**, sem pré-treinamento. Linha de base inferior — mostra o custo de não usar Transfer Learning.

In [5]:
torch.manual_seed(42)
m = models.resnet18(weights=None)
m.fc = nn.Linear(m.fc.in_features, 10)
m = m.to(device)
opt = optim.SGD(m.parameters(), lr=0.01, momentum=0.9)
run_experiment('Scratch Padrão', m, opt, train_loader_std)


  Scratch Padrão
  Parâmetros treináveis: 11,181,642
  Época  1/10 | Loss: 2.0133 | Val Acc: 25.90%
  Época  2/10 | Loss: 1.7095 | Val Acc: 31.10%
  Época  3/10 | Loss: 1.4734 | Val Acc: 37.60%
  Época  4/10 | Loss: 1.3088 | Val Acc: 42.50%
  Época  5/10 | Loss: 1.1242 | Val Acc: 46.20%
  Época  6/10 | Loss: 0.9403 | Val Acc: 44.40%
  Época  7/10 | Loss: 0.7307 | Val Acc: 38.70%
  Época  8/10 | Loss: 0.5682 | Val Acc: 43.70%
  Época  9/10 | Loss: 0.3690 | Val Acc: 46.50%
  Época 10/10 | Loss: 0.2065 | Val Acc: 42.00%
  ✅ 2.9 min | Acurácia final: 42.00%


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Experimento 2 — Scratch + Data Augmentation
Mesmo modelo do zero, mas com flip horizontal e rotação aleatória no treino. Avalia se regularização por augmentation compensa a ausência de pré-treinamento.

In [6]:
torch.manual_seed(42)
m = models.resnet18(weights=None)
m.fc = nn.Linear(m.fc.in_features, 10)
m = m.to(device)
opt = optim.SGD(m.parameters(), lr=0.01, momentum=0.9)
run_experiment('Scratch + Augmentation', m, opt, train_loader_aug)


  Scratch + Augmentation
  Parâmetros treináveis: 11,181,642
  Época  1/10 | Loss: 2.0303 | Val Acc: 25.20%
  Época  2/10 | Loss: 1.7592 | Val Acc: 34.00%
  Época  3/10 | Loss: 1.5781 | Val Acc: 36.40%
  Época  4/10 | Loss: 1.4968 | Val Acc: 40.40%
  Época  5/10 | Loss: 1.3810 | Val Acc: 45.40%
  Época  6/10 | Loss: 1.3296 | Val Acc: 49.70%
  Época  7/10 | Loss: 1.2138 | Val Acc: 40.50%
  Época  8/10 | Loss: 1.2041 | Val Acc: 51.50%
  Época  9/10 | Loss: 1.0872 | Val Acc: 48.90%
  Época 10/10 | Loss: 1.0102 | Val Acc: 52.20%
  ✅ 2.9 min | Acurácia final: 52.20%


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Experimento 3 — Feature Extraction
Backbone **congelado** (pesos ImageNet), treina apenas a camada `fc` (5.130 parâmetros). Estratégia ensinada no Lab 1 e no Colab.

In [7]:
torch.manual_seed(42)
m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for p in m.parameters():
    p.requires_grad = False
m.fc = nn.Linear(m.fc.in_features, 10)
m = m.to(device)
opt = optim.SGD(m.fc.parameters(), lr=0.01, momentum=0.9)
run_experiment('Feature Extraction', m, opt, train_loader_std)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 132MB/s]



  Feature Extraction
  Parâmetros treináveis: 5,130
  Época  1/10 | Loss: 1.1330 | Val Acc: 71.70%
  Época  2/10 | Loss: 0.6711 | Val Acc: 75.50%
  Época  3/10 | Loss: 0.6086 | Val Acc: 75.90%
  Época  4/10 | Loss: 0.5529 | Val Acc: 76.60%
  Época  5/10 | Loss: 0.5413 | Val Acc: 71.30%
  Época  6/10 | Loss: 0.5331 | Val Acc: 75.20%
  Época  7/10 | Loss: 0.5054 | Val Acc: 72.50%
  Época  8/10 | Loss: 0.5068 | Val Acc: 75.60%
  Época  9/10 | Loss: 0.4715 | Val Acc: 75.00%
  Época 10/10 | Loss: 0.4548 | Val Acc: 76.10%
  ✅ 1.9 min | Acurácia final: 76.10%


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Experimento 4 — Fine-Tuning Parcial (layer4)
Descongela apenas o último bloco residual (`layer4`) com taxa de aprendizado baixa, mantendo as primeiras camadas congeladas. Estratégia do Lab 2.

In [8]:
torch.manual_seed(42)
m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for p in m.parameters():
    p.requires_grad = False
for p in m.layer4.parameters():
    p.requires_grad = True
m.fc = nn.Linear(m.fc.in_features, 10)
m = m.to(device)
opt = optim.SGD(
    [{'params': m.layer4.parameters(), 'lr': 1e-4},
     {'params': m.fc.parameters(),    'lr': 1e-3}],
    momentum=0.9)
run_experiment('Fine-Tuning Parcial (layer4)', m, opt, train_loader_std)


  Fine-Tuning Parcial (layer4)
  Parâmetros treináveis: 8,398,858
  Época  1/10 | Loss: 1.8591 | Val Acc: 62.30%
  Época  2/10 | Loss: 1.1257 | Val Acc: 70.40%
  Época  3/10 | Loss: 0.8810 | Val Acc: 75.30%
  Época  4/10 | Loss: 0.7533 | Val Acc: 75.50%
  Época  5/10 | Loss: 0.6758 | Val Acc: 76.50%
  Época  6/10 | Loss: 0.6165 | Val Acc: 78.80%
  Época  7/10 | Loss: 0.5721 | Val Acc: 78.10%
  Época  8/10 | Loss: 0.5414 | Val Acc: 79.80%
  Época  9/10 | Loss: 0.4973 | Val Acc: 79.70%
  Época 10/10 | Loss: 0.4715 | Val Acc: 79.30%
  ✅ 2.0 min | Acurácia final: 79.30%


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Experimento 5 — Fine-Tuning Completo
Todas as 18 camadas abertas para atualização, com taxa de aprendizado baixa para evitar catastrophic forgetting. Teto de desempenho com este dataset.

In [9]:
torch.manual_seed(42)
m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
m.fc = nn.Linear(m.fc.in_features, 10)
m = m.to(device)
opt = optim.SGD(m.parameters(), lr=0.001, momentum=0.9)
run_experiment('Fine-Tuning Completo', m, opt, train_loader_std)


  Fine-Tuning Completo
  Parâmetros treináveis: 11,181,642
  Época  1/10 | Loss: 1.5533 | Val Acc: 72.90%
  Época  2/10 | Loss: 0.6393 | Val Acc: 83.70%
  Época  3/10 | Loss: 0.3916 | Val Acc: 86.40%
  Época  4/10 | Loss: 0.2490 | Val Acc: 88.20%
  Época  5/10 | Loss: 0.1703 | Val Acc: 88.40%
  Época  6/10 | Loss: 0.1126 | Val Acc: 88.10%
  Época  7/10 | Loss: 0.0806 | Val Acc: 89.10%
  Época  8/10 | Loss: 0.0693 | Val Acc: 89.30%
  Época  9/10 | Loss: 0.0417 | Val Acc: 88.90%
  Época 10/10 | Loss: 0.0380 | Val Acc: 88.70%
  ✅ 3.0 min | Acurácia final: 88.70%


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Tabela de Resultados

In [10]:
print(f'\n{"="*65}')
print(f'{"BENCHMARK COMPLETO — CIFAR-10 · 5.000 imagens · 10 épocas":^65}')
print(f'{"="*65}')
print(f'{"Experimento":<30} {"Params":>12} {"Acurácia":>10} {"Tempo":>10}')
print(f'{"-"*65}')
for name, r in results.items():
    print(f'{name:<30} {r["params"]:>12,} {r["acc"]:>9.2f}% {r["time_min"]:>8.1f}min')
print(f'{"="*65}')
best = max(results, key=lambda k: results[k]['acc'])
print(f'  Melhor: {best} → {results[best]["acc"]:.2f}%')


    BENCHMARK COMPLETO — CIFAR-10 · 5.000 imagens · 10 épocas    
Experimento                          Params   Acurácia      Tempo
-----------------------------------------------------------------
Scratch Padrão                   11,181,642     42.00%      2.9min
Scratch + Augmentation           11,181,642     52.20%      2.9min
Feature Extraction                    5,130     76.10%      1.9min
Fine-Tuning Parcial (layer4)      8,398,858     79.30%      2.0min
Fine-Tuning Completo             11,181,642     88.70%      3.0min
  Melhor: Fine-Tuning Completo → 88.70%
